In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, model_validator, Field

class Generator(BaseModel):
    name: str
    at_bus: str
    cost_0: float = 0.0
    cost_1: float | None = None
    cost_2: float = 0.0
    max_capacity_mw: float | None = Field(default=None, description="Maximum generation capacity.")
    min_capacity_mw: float = 0.0

class Bus(BaseModel):
    name: str
    load_mw: float

class Line(BaseModel):
    name: str
    from_bus: str | None = None
    to_bus: str | None = None
    max_flow_mw: float | None = Field(default=None, description="The maximum power flow, limit, or capacity of the line.")
    reactance: float | None = Field(default=None, description="The reactance or X value of the line.")


class DC_OPF_Problem(BaseModel):
    objective_type: str
    generators: list[Generator]
    lines: list[Line]
    buses: list[Bus]

    @model_validator(mode='after')
    def check_grid_physics_and_topology(self) -> 'DC_OPF_Problem':
        # 1. Topology Check: Do the lines connect to real buses?
        for line in self.lines:
            if line.reactance is None or line.max_flow_mw is None:
                raise ValueError(f"Missing Data Error: Line '{line.name}' is missing reactance or max flow!")
                
        for gen in self.generators:
            if None in (gen.cost_1, gen.max_capacity_mw):
                raise ValueError(f"Missing Data Error: Generator '{gen.name}' is missing cost or capacity data!")
            
        bus_names = {b.name for b in self.buses}
        for line in self.lines:
            if line.from_bus not in bus_names:
                raise ValueError(f"Topology Error: Line '{line.name}' starts at '{line.from_bus}', but that bus doesn't exist!")
            if line.to_bus not in bus_names:
                raise ValueError(f"Topology Error: Line '{line.name}' ends at '{line.to_bus}', but that bus doesn't exist!")

        # 2. Basic Physics Check: Is there enough generation to meet demand?
        total_load = sum(b.load_mw for b in self.buses)
        total_capacity = sum(g.max_capacity_mw for g in self.generators)
        
        if total_capacity < total_load:
            raise ValueError(f"Physics Error: Grid is doomed! Total Load ({total_load} MW) exceeds Total Max Capacity ({total_capacity} MW).")
        
        # 3. Non-Negative Value Check
        for g in self.generators:
            if g.max_capacity_mw < 0 or g.min_capacity_mw < 0:
                raise ValueError(f"Value Error: Generator '{g.name}' has negative capacity!")
        
        for line in self.lines:
            if line.reactance < 0: 
                raise ValueError(f"Value Error: line '{line.name}' has negative reactance!")
            if line.max_flow_mw < 0:
                raise ValueError(f"Value Error: line '{line.name}' has negative flow capacity!")
        
        return self

In [2]:
# 1. Load the hidden API key from your .env file
load_dotenv()

# 2. Initialize the Gemini client 
client = genai.Client()

strict_instructions = (
        "You are a strict grid data extraction system. "
        "CRITICAL RULE: If a numeric value (like reactance, cost, or max capacity) "
        "is not explicitly stated in the text, DO NOT guess, DO NOT calculate it, "
        "and DO NOT assume a default value of 0. You MUST set the value to null. \n\n"
        "Grid Text to Extract:\n"
    )

# 3. Define the plain-text problem (Unstructured Data)
prompt_text = """
Run OPF for a simple 2-bus corridor. Bus A is the reference.

Bus A has Generator Alpha: No-load cost $200, linear cost $50/MW, quadratic 0.15. Capacity: 0 to 300 MW.
Bus B has 250 MW of load and Generator Beta: No-load cost $100, linear cost $60/MW, quadratic 0.2. Capacity: 0 to 200 MW.

A single transmission line connects Bus A to Bus B. Reactance 0.1, flow limit 150 MW.
"""

full_prompt = strict_instructions + prompt_text


# 4. Ask Gemini to read the text and fill out the Pydantic form
print("Sending text to Gemini...")
response = client.models.generate_content(
    model='gemini-3.1-flash-lite-preview',
    contents=full_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DC_OPF_Problem,
        temperature=0.0,
    ),
)

# 5. Convert the JSON response back into our Pydantic object
extracted_data = DC_OPF_Problem.model_validate_json(response.text)

print("\n--- Extraction Success! ---")
print(f"Objective: {extracted_data.objective_type}")
for gen in extracted_data.generators:
    print(f"Found {gen.name}: Min {gen.min_capacity_mw}MW, Max {gen.max_capacity_mw}MW, Cost {gen.cost_2}*P^2 + {gen.cost_1}*P + {gen.cost_0}, at {gen.at_bus}")
for bus in extracted_data.buses:
    print(f"Found {bus.name}: laod {bus.load_mw}MW")
for l in extracted_data.lines:
    print(f"Found {l.name}: From {l.from_bus} to {l.to_bus}, Max Flow {l.max_flow_mw}MW, Reactance {l.reactance}")

Sending text to Gemini...

--- Extraction Success! ---
Objective: OPF
Found Generator Alpha: Min 0.0MW, Max 300.0MW, Cost 0.15*P^2 + 50.0*P + 200.0, at Bus A
Found Generator Beta: Min 0.0MW, Max 200.0MW, Cost 0.2*P^2 + 60.0*P + 100.0, at Bus B
Found Bus A: laod 0.0MW
Found Bus B: laod 250.0MW
Found Line A-B: From Bus A to Bus B, Max Flow 150.0MW, Reactance 0.1


In [20]:
extracted_data

DC_OPF_Problem(objective_type='OPF', generators=[Generator(name='Generator A', at_bus='Bus 1', cost_0=30.0, cost_1=0.0, cost_2=0.0, max_capacity_mw=200.0, min_capacity_mw=0.0), Generator(name='Generator B', at_bus='Bus 3', cost_0=0.0, cost_1=0.0, cost_2=0.0, max_capacity_mw=150.0, min_capacity_mw=0.0)], lines=[Line(name='Line 1-2', from_bus='Bus 1', to_bus='Bus 2', max_flow_mw=100.0, reactance=0.0), Line(name='Line 2-3', from_bus='Bus 2', to_bus='Bus 3', max_flow_mw=100.0, reactance=0.0), Line(name='Line 1-3', from_bus='Bus 1', to_bus='Bus 3', max_flow_mw=100.0, reactance=0.0)], buses=[Bus(name='Bus 1', load_mw=0.0), Bus(name='Bus 2', load_mw=100.0), Bus(name='Bus 3', load_mw=0.0)])

In [54]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()

# extract the paramters
model.generator = pyo.Set(initialize = [g.name for g in extracted_data.generators])
model.line = pyo.Set(initialize = [l.name for l in extracted_data.lines])
model.bus = pyo.Set(initialize= [b.name for b in extracted_data.buses])

## parameters of generator
model.gen_power = pyo.Var(model.generator)

costs = {g.name: [g.cost_0, g.cost_1, g.cost_2] for g in extracted_data.generators}
model.cost = pyo.Param(model.generator, initialize=costs)
model.theta = pyo.Var(model.bus, domain=pyo.Reals)
gen_loc = {g.name: g.at_bus for g in extracted_data.generators}
model.gen_loc = pyo.Param(model.generator, initialize=gen_loc)

min_capacity = {g.name: g.min_capacity_mw for g in extracted_data.generators}
model.min_capicity = pyo.Param(model.generator, initialize=min_capacity)
max_capacity = {g.name: g.max_capacity_mw for g in extracted_data.generators}
model.max_capicity = pyo.Param(model.generator, initialize=max_capacity)

# parameters of line
from_bus = {l.name: l.from_bus for l in extracted_data.lines}
to_bus = {l.name: l.to_bus for l in extracted_data.lines}
model.line_from = pyo.Param(model.line, initialize=from_bus)
model.line_to = pyo.Param(model.line, initialize=to_bus)

susceptance = {l.name: 1/l.reactance for l in extracted_data.lines}
model.B = pyo.Param(model.line, initialize=susceptance)

max_flow = {l.name: l.max_flow_mw for l in extracted_data.lines}
model.max_flow = pyo.Param(model.line, initialize=max_flow)

# parameters of bus
bus_load = {b.name: b.load_mw for b in extracted_data.buses}
model.bus_load = pyo.Param(model.bus, initialize=bus_load)


# Constrainsts
def max_rule(model, g): # generator limit
    return model.gen_power[g] <= model.max_capicity[g]
def min_rule(model, g):
    return model.gen_power[g] >= model.min_capicity[g]
model.c1 = pyo.Constraint(model.generator, rule = max_rule)
model.c2 = pyo.Constraint(model.generator, rule = min_rule)

model.ref_angle = pyo.Constraint(expr= model.theta[model.bus.first()] == 0) # set the first bus as reference

def calc_line_flow(model, line):
    start_bus = model.line_from[line]
    end_bus = model.line_to[line]
    return model.B[line] * (model.theta[start_bus] - model.theta[end_bus])
model.line_flow = pyo.Expression(model.line, rule=calc_line_flow)

def line_limit_rule(model, line): # line flow limit
    return (-model.max_flow[line], model.line_flow[line], model.max_flow[line])
model.line_limits = pyo.Constraint(model.line, rule=line_limit_rule)

def Nodal_rule(model, bus): # nodal balance
    generator_mw = 0
    for g in model.generator:
        if model.gen_loc[g] == bus:
            generator_mw += model.gen_power[g]
    load_mw = model.bus_load[bus]
    flow_in = sum(model.line_flow[l] for l in model.line if model.line_to[l] == bus)
    flow_out = sum(model.line_flow[l] for l in model.line if model.line_from[l] == bus)
    return (generator_mw - load_mw) == (flow_out - flow_in)
model.nodal_balance = pyo.Constraint(model.bus, rule=Nodal_rule)

# Objective function
model.f1 = pyo.Objective(expr = sum(model.cost[g][2] * model.gen_power[g]**2 + model.cost[g][1] * model.gen_power[g] + model.cost[g][0] for g in model.generator), sense=pyo.minimize)

# Solve
solver = pyo.SolverFactory('ipopt')
results = solver.solve(model)
for g in model.generator:
    print(f"{g}: {pyo.value(model.gen_power[g])} MW")

North Hub Coal: 480.5555590831936 MW
Sunny Suburb Solar: 100.00000099996663 MW
City Center Gas Peaker: 119.44443991683981 MW
